# HoloMine v2. MAM CLIP ConvNeXt full fine tune, EMA, TTA, dan koreksi prior shift

Pendekatan yang disesuaikan dari dua notebook sebelumnya.

1. Backbone ConvNeXt dari MAM CLIP. Default memakai `convnext_tiny_ftune.pth`, yaitu vision encoder yang sudah di fine tune untuk BI RADS. Bisa diganti ke checkpoint pretrain atau ke ConvNeXt Small lewat konfigurasi.
2. Seluruh backbone dilatih dengan layer wise LR decay, warmup lalu cosine, label smoothing, mixup ringan, dan horizontal flip.
3. Tidak ada pemilihan epoch terbaik dari validasi. Bobot yang dipakai adalah EMA di akhir jadwal, sehingga angka OOF tidak optimis.
4. Inferensi memakai TTA horizontal flip dan rata rata 5 fold.
5. Prior test diestimasi secara unsupervised dengan EM (Saerens 2002) lalu dipakai untuk mengoreksi probabilitas. Ini menjawab pergeseran distribusi kelas antara train (80/52/80) dan test.
6. Jika file ground truth di attach, skor test dihitung untuk tiga skenario sebagai bahan belajar. Ground truth tidak pernah dipakai untuk training atau tuning.

Referensi bobot. Gulluk dan Gevaert, MAM CLIP, arXiv 2605.19359, repo igulluk/MAM-CLIP.

## 1. Konfigurasi

In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('holomine-breasts-cancer-classification-task-1')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/holomine-breasts-cancer-classification-task-1


In [4]:
from pathlib import Path

FAST_DEBUG = False      # True untuk uji jalan (1 fold, 1 epoch, 5 citra per kelas)
SEED = 42

# Setiap run = 5 fold. Tambah run untuk ensemble lebih beragam, misalnya init "pretrain" atau model "convnext_small".
RUN_SPECS = [
    {"name": "main_tiny_ftune", "model": "convnext_tiny", "init": "ftune", "seed": 42, "task": "3class"},
    {"name": "spec_tiny_ftune", "model": "convnext_tiny", "init": "ftune", "seed": 42, "task": "bvm"},
    # {"name": "main_tiny_pretrain", "model": "convnext_tiny", "init": "pretrain", "seed": 43, "task": "3class"},
    # {"name": "main_small_ftune", "model": "convnext_small", "init": "ftune", "seed": 44, "task": "3class"},
]
TILE_ENCODER = {"model": "convnext_tiny", "init": "ftune"}   # encoder beku untuk embedding tile

# Folder artifacts dari run lama (misalnya v2) yang berisi oof_<nama>.csv dan test_<nama>.csv.
# Unggah folder artifacts v2 sebagai Kaggle Dataset lalu isi path nya di sini supaya modelnya ikut di ensemble tanpa dilatih ulang.
EXTERNAL_ARTIFACT_DIRS = []   # contoh: [Path("/kaggle/input/holomine-v2-artifacts")]

CLASS_NAMES = ["Normal", "Benign", "Malignant"]
LABEL_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {i: n for n, i in LABEL_TO_ID.items()}

DATA_ROOT = Path("/kaggle/input/competitions/holomine-breasts-cancer-classification-task-1")
TRAIN_CSV, TEST_CSV = DATA_ROOT / "train.csv", DATA_ROOT / "test.csv"
SAMPLE_SUBMISSION_CSV = DATA_ROOT / "sample_submission.csv"
TRAIN_IMAGE_DIR, TEST_IMAGE_DIR = DATA_ROOT / "train_images", DATA_ROOT / "test_images"

OUTPUT_DIR = Path("/kaggle/working")
WEIGHT_DIR, ARTIFACT_DIR = OUTPUT_DIR / "mamclip_weights", OUTPUT_DIR / "artifacts"
for d in (WEIGHT_DIR, ARTIFACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

IMAGE_HEIGHT, IMAGE_WIDTH = 1024, 768   # input CNN
HIRES_HEIGHT, HIRES_WIDTH = 2048, 1536  # sumber tile dan fitur tabular
TILE = 512
EMBED_DIM = 768

EPOCHS, WARMUP_EPOCHS = 25, 1
BATCH_SIZE, GRAD_ACCUM_STEPS, NUM_WORKERS = 4, 2, 2
BASE_LR, HEAD_LR, LLRD, WEIGHT_DECAY = 5e-5, 5e-4, 0.7, 0.05
LABEL_SMOOTHING, MIXUP_ALPHA, MIXUP_PROB, EMA_DECAY, DROP_PATH = 0.1, 0.2, 0.3, 0.998, 0.1
TTA_HFLIP = True
DEBUG_SAMPLES_PER_CLASS, DEBUG_EPOCHS = 5, 1

## 2. Unduh bobot MAM CLIP (hanya file yang dibutuhkan)

In [5]:
import importlib.util, subprocess, sys

GDRIVE_IDS = {
    ("convnext_tiny", "ftune"): ("1AduR91COdEtDS4D55ZxkdRFmwW_x6nPA", "convnext_tiny_ftune.pth"),
    ("convnext_tiny", "pretrain"): ("1SHiUSnSM9yINdLt-0t-RDTxVBUsleAEA", "pretrain_convnext_tiny.ckpt"),
    ("convnext_small", "ftune"): ("1MHMaYD-F3Vk05sCh0J3OOWGAa1-j57ES", "convnext_small_ftune.pth"),
    ("convnext_small", "pretrain"): ("1Luuc5_U9yCBhDMH2_z2aU6M17zgcNXGl", "pretrain_convnext_small.ckpt"),
}


def get_weight_path(model_name, init_mode):
    file_id, file_name = GDRIVE_IDS[(model_name, init_mode)]
    found = list(Path("/kaggle/input").rglob(file_name)) + list(WEIGHT_DIR.rglob(file_name))
    if found:
        return found[0]
    if importlib.util.find_spec("gdown") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "gdown"])
    import gdown
    out = WEIGHT_DIR / file_name
    gdown.download(id=file_id, output=str(out), quiet=False, use_cookies=False)
    assert out.is_file(), out
    return out


needed = {(s["model"], s["init"]) for s in RUN_SPECS} | {(TILE_ENCODER["model"], TILE_ENCODER["init"])}
WEIGHT_PATHS = {k: get_weight_path(*k) for k in needed}
for k, v in WEIGHT_PATHS.items():
    print(k, v, f"{v.stat().st_size / 1024**2:.0f} MiB")

('convnext_tiny', 'ftune') /kaggle/working/mamclip_weights/convnext_tiny_ftune.pth 108 MiB


## 3. Import dan reproducibility

In [6]:
import copy, json, math, os, random, warnings
from contextlib import nullcontext

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import stats as sstats
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print(DEVICE, "| timm", timm.__version__, "| albumentations", A.__version__)
if not AMP:
    warnings.warn("GPU tidak terdeteksi.")

cuda | timm 1.0.26 | albumentations 2.0.8


## 4. Validasi CSV

In [7]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
assert set(train_df["label"]) <= set(CLASS_NAMES) and not train_df["image_id"].duplicated().any()
assert set(test_df["image_id"].astype(str)) == set(sample_df["image_id"].astype(str))
train_df["image_id"] = train_df["image_id"].astype(str)
train_df["target"] = train_df["label"].map(LABEL_TO_ID).astype(int)
TEST_IDS = sample_df["image_id"].astype(str).tolist()

if FAST_DEBUG:
    train_df = train_df.groupby("target", group_keys=False).sample(n=DEBUG_SAMPLES_PER_CLASS, random_state=SEED).reset_index(drop=True)
N_TRAIN = len(train_df)
Y_TRAIN = train_df["target"].to_numpy()

train_df["fold"] = -1
for f, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(train_df, Y_TRAIN)):
    train_df.loc[va, "fold"] = f
FOLDS = [0] if FAST_DEBUG else list(range(5))
EPOCHS_RUN = DEBUG_EPOCHS if FAST_DEBUG else EPOCHS
print(train_df["label"].value_counts().reindex(CLASS_NAMES).to_dict(), "| test", len(TEST_IDS))
print(train_df.groupby("fold")["label"].value_counts().unstack().reindex(columns=CLASS_NAMES))

{'Normal': 80, 'Benign': 52, 'Malignant': 80} | test 54
label  Normal  Benign  Malignant
fold                            
0          16      11         16
1          16      11         16
2          16      10         16
3          16      10         16
4          16      10         16


## 5. Crop area payudara dan cache grayscale

In [8]:
def resolve_paths(ids, image_dir):
    index = {p.name: p for p in image_dir.rglob("*") if p.is_file()}
    out = {}
    for iid in ids:
        p = image_dir / iid
        out[iid] = p if p.is_file() else index[Path(iid).name]
    return out


def read_gray(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"))


def crop_breast(gray, margin=0.03):
    h, w = gray.shape
    if gray.max() <= gray.min():
        return gray
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    otsu, _ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (blur > max(3, int(0.5 * otsu))).astype(np.uint8) * 255
    k = max(3, int(min(h, w) * 0.01)) | 1
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_CLOSE, ker), cv2.MORPH_OPEN, ker)
    n, _, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return gray
    c = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    x, y, bw, bh, area = st[c]
    if area < 0.02 * h * w:
        return gray
    mx, my = max(2, int(w * margin)), max(2, int(h * margin))
    x0, y0, x1, y1 = max(0, x - mx), max(0, y - my), min(w, x + bw + mx), min(h, y + bh + my)
    if x <= 0.05 * w: x0 = 0
    if x + bw >= 0.95 * w: x1 = w
    out = gray[y0:y1, x0:x1]
    return out if out.size else gray


def breast_mask(gray):
    otsu, _ = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray > max(3, int(0.5 * otsu))


def tabular_features(hires):
    # hires = grayscale uint8 2048x1536 hasil crop
    m = breast_mask(hires)
    px = hires[m].astype(np.float32)
    if px.size < 100:
        px = hires.astype(np.float32).ravel()
    feats = {"area_frac": float(m.mean()), "mean": float(px.mean()), "std": float(px.std()),
             "skew": float(sstats.skew(px)), "kurt": float(sstats.kurtosis(px))}
    for q in (10, 25, 50, 75, 90):
        feats[f"p{q}"] = float(np.percentile(px, q))
    t, _ = cv2.threshold(px.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    feats["dense_frac"] = float((px > t).mean())
    feats["bright_frac"] = float((px > np.percentile(px, 95)).mean())
    # kandidat mikrokalsifikasi. top hat kernel kecil lalu komponen kecil terang
    th = cv2.morphologyEx(hires, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    cand = ((th > max(20, np.percentile(th[m], 99.5))) & m).astype(np.uint8)
    n, _, st, _ = cv2.connectedComponentsWithStats(cand, connectivity=8)
    areas = st[1:, cv2.CC_STAT_AREA] if n > 1 else np.array([])
    small = areas[(areas >= 2) & (areas <= 40)]
    feats["mc_count"] = float(len(small)); feats["mc_density"] = float(len(small) / max(1.0, m.sum() / 1e5))
    feats["tophat_p99"] = float(np.percentile(th[m], 99)) if m.any() else 0.0
    # tekstur pada 512x384
    small_img = cv2.resize(hires, (384, 512), interpolation=cv2.INTER_AREA)
    g = (small_img // 8).astype(np.uint8)
    glcm = graycomatrix(g, distances=[1, 3], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=32, symmetric=True, normed=True)
    for prop in ("contrast", "homogeneity", "energy", "correlation"):
        feats[f"glcm_{prop}"] = float(graycoprops(glcm, prop).mean())
    lbp = local_binary_pattern(small_img, P=8, R=1, method="uniform")
    hist = np.bincount(lbp[cv2.resize(m.astype(np.uint8), (384, 512)) > 0].astype(int), minlength=10)[:10]
    hist = hist / max(1, hist.sum())
    for i, v in enumerate(hist):
        feats[f"lbp_{i}"] = float(v)
    return feats


def make_tiles(hires):
    # 4 x 3 tile 512 dari 2048x1536, plus fraksi piksel payudara tiap tile
    m = breast_mask(hires)
    tiles, fracs = [], []
    for r in range(HIRES_HEIGHT // TILE):
        for c in range(HIRES_WIDTH // TILE):
            ys, xs = slice(r * TILE, (r + 1) * TILE), slice(c * TILE, (c + 1) * TILE)
            tiles.append(hires[ys, xs]); fracs.append(float(m[ys, xs].mean()))
    return np.stack(tiles), np.array(fracs, dtype=np.float32)


train_paths = resolve_paths(train_df["image_id"], TRAIN_IMAGE_DIR)
test_paths = resolve_paths(TEST_IDS, TEST_IMAGE_DIR)
cache_cnn, cache_tiles, tile_fracs, tab_rows = {}, {}, {}, {}
for iid, path in tqdm(list(train_paths.items()) + list(test_paths.items()), desc="Preprocess"):
    g = crop_breast(read_gray(path))
    cache_cnn[iid] = cv2.resize(g, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_AREA)
    hires = cv2.resize(g, (HIRES_WIDTH, HIRES_HEIGHT), interpolation=cv2.INTER_AREA)
    cache_tiles[iid], tile_fracs[iid] = make_tiles(hires)
    tab_rows[iid] = tabular_features(hires)
    del hires
tab_df = pd.DataFrame.from_dict(tab_rows, orient="index")
print("fitur tabular", tab_df.shape, "| tile per citra", cache_tiles[TEST_IDS[0]].shape)

Preprocess:   0%|          | 0/266 [00:00<?, ?it/s]

fitur tabular (266, 29) | tile per citra (12, 512, 512)


## 6. Model. Vision encoder MAM CLIP plus head 3 kelas

In [9]:
def load_encoder(model_name, init_mode):
    enc = timm.create_model(model_name, pretrained=False, drop_path_rate=DROP_PATH)
    enc.head.fc = nn.Linear(enc.head.fc.in_features, EMBED_DIM)
    raw = torch.load(WEIGHT_PATHS[(model_name, init_mode)], map_location="cpu", weights_only=False)
    if init_mode == "pretrain":
        pre = "VLModel.vision_encoder."
        state = {k[len(pre):]: v for k, v in raw["state_dict"].items() if k.startswith(pre)}
    else:
        state = {k: v for k, v in raw.items() if not k.startswith(("birads_layer", "density_layer"))}
    res = enc.load_state_dict(state, strict=False)
    missing = set(res.missing_keys) & {n for n, _ in enc.named_parameters()}
    assert not missing, sorted(missing)[:5]
    return enc


class Classifier(nn.Module):
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder, self.drop, self.fc = encoder, nn.Dropout(0.2), nn.Linear(EMBED_DIM, n_classes)

    def forward(self, x):
        return self.fc(self.drop(F.gelu(self.encoder(x))))


def make_aug():
    v2 = int(A.__version__.split(".")[0]) >= 2
    aff = dict(scale=(0.9, 1.1), rotate=(-10, 10), translate_percent=(0.0, 0.03), p=0.8)
    affine = A.Affine(border_mode=cv2.BORDER_CONSTANT, fill=0, **aff) if v2 else A.Affine(mode=cv2.BORDER_CONSTANT, cval=0, **aff)
    noise = A.GaussNoise(std_range=(0.005, 0.02), p=0.25) if v2 else A.GaussNoise(var_limit=(2.0, 20.0), p=0.25)
    return A.Compose([A.HorizontalFlip(p=0.5), affine, A.RandomBrightnessContrast(0.1, 0.1, p=0.4),
                      A.RandomGamma(gamma_limit=(85, 115), p=0.3), noise])


AUG = make_aug()


def to_tensor(gray):
    t = torch.from_numpy(np.ascontiguousarray(gray)).float()
    t = t - t.amin(); mx = t.amax()
    t = t / mx if mx > 0 else t
    return t.unsqueeze(0).repeat(3, 1, 1)


class DS(Dataset):
    def __init__(self, ids, targets, train):
        self.ids, self.targets, self.train = list(ids), None if targets is None else list(targets), train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        g = cache_cnn[self.ids[i]]
        if self.train:
            g = AUG(image=np.repeat(g[:, :, None], 3, axis=2))["image"][:, :, 0]
        y = -1 if self.targets is None else self.targets[i]
        return to_tensor(g), y, self.ids[i]


def loader(ids, targets, train, seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(DS(ids, targets, train), batch_size=BATCH_SIZE, shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=True, generator=g, persistent_workers=NUM_WORKERS > 0)


class EMA:
    def __init__(self, model, decay):
        self.m, self.d = copy.deepcopy(model).eval(), decay
        for p in self.m.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        sd = model.state_dict()
        for k, v in self.m.state_dict().items():
            v.mul_(self.d).add_(sd[k].detach(), alpha=1 - self.d) if v.dtype.is_floating_point else v.copy_(sd[k])


def param_groups(model):
    enc, groups = model.encoder, []
    ns = len(enc.stages)

    def add(params, lr):
        dec = [p for p in params if p.requires_grad and p.ndim > 1]
        nod = [p for p in params if p.requires_grad and p.ndim <= 1]
        if dec: groups.append({"params": dec, "lr": lr, "weight_decay": WEIGHT_DECAY})
        if nod: groups.append({"params": nod, "lr": lr, "weight_decay": 0.0})

    add(list(enc.stem.parameters()), BASE_LR * LLRD ** ns)
    for i, st in enumerate(enc.stages):
        add(list(st.parameters()), BASE_LR * LLRD ** (ns - 1 - i))
    add([p for n, p in enc.named_parameters() if not n.startswith(("stem", "stages"))], BASE_LR)
    add(list(model.fc.parameters()), HEAD_LR)
    return groups


def ac():
    return torch.autocast("cuda", dtype=torch.float16) if AMP else nullcontext()


@torch.inference_mode()
def predict(model, ids):
    model.eval(); out = []
    for x, _, _ in loader(ids, None, False, 0):
        x = x.to(DEVICE, non_blocking=True)
        with ac():
            lg = model(x).float()
            if TTA_HFLIP:
                lg = 0.5 * (lg + model(torch.flip(x, dims=[3])).float())
        out.append(torch.softmax(lg, 1).cpu().numpy())
    return np.concatenate(out)


def train_fold(spec, tr_ids, tr_y, n_classes, seed):
    seed_everything(seed)
    model = Classifier(load_encoder(spec["model"], spec["init"]), n_classes).to(DEVICE)
    ema = EMA(model, EMA_DECAY)
    cnt = np.bincount(tr_y, minlength=n_classes).astype(float)
    w = torch.tensor((1 / cnt) / (1 / cnt).mean(), dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(param_groups(model))
    dl = loader(tr_ids, tr_y, True, seed)
    spe = math.ceil(len(dl) / GRAD_ACCUM_STEPS)
    total, warm = spe * EPOCHS_RUN, spe * WARMUP_EPOCHS
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    for ep in range(EPOCHS_RUN):
        model.train(); run = 0.0; opt.zero_grad(set_to_none=True)
        for step, (x, y, _) in enumerate(dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            mix = random.random() < MIXUP_PROB and x.size(0) > 1
            if mix:
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)); perm = torch.randperm(x.size(0), device=DEVICE)
                x = lam * x + (1 - lam) * x[perm]
            with ac():
                lg = model(x)
                loss = lam * crit(lg, y) + (1 - lam) * crit(lg, y[perm]) if mix else crit(lg, y)
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(dl):
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step(); ema.update(model)
            run += loss.item() * x.size(0)
        print(f"  epoch {ep + 1:02d}/{EPOCHS_RUN} loss {run / len(dl.dataset):.4f}")
    del model, opt, dl
    return ema.m

## 7. Dataset, augmentasi, normalisasi resmi

In [10]:
def macro(y, p):
    return f1_score(y, p.argmax(1), average="macro", zero_division=0)


OOF, TEST = {}, {}   # name -> array
for spec in RUN_SPECS:
    n_cls = 3 if spec["task"] == "3class" else 2
    oof = np.full((N_TRAIN, n_cls), np.nan, dtype=np.float32); tst = []
    print(f"\n######## {spec['name']} ({spec['task']}) ########")
    for f in FOLDS:
        tr = train_df[train_df["fold"] != f]; va = train_df[train_df["fold"] == f]
        if spec["task"] == "bvm":
            tr = tr[tr["target"] > 0]
            y_tr = (tr["target"].to_numpy() == 1).astype(int)   # 1 = Benign, 0 = Malignant
        else:
            y_tr = tr["target"].to_numpy()
        print(f"fold {f} | train {len(tr)}")
        model = train_fold(spec, tr["image_id"].tolist(), y_tr, n_cls, spec["seed"] + f)
        oof[va.index.to_numpy()] = predict(model, va["image_id"].tolist())
        tst.append(predict(model, TEST_IDS))
        if spec["task"] == "3class":
            print(f"  fold {f} EMA macro F1 {macro(va['target'].to_numpy(), oof[va.index]):.4f}")
        else:
            ab = va[va["target"] > 0]
            print(f"  fold {f} spesialis F1 (Benign vs Malignant) {f1_score((ab['target'] == 1).astype(int), oof[ab.index].argmax(1), average='macro'):.4f}")
        del model; torch.cuda.empty_cache()
    OOF[spec["name"]], TEST[spec["name"]] = oof, np.mean(tst, axis=0)
    np.save(ARTIFACT_DIR / f"oof_{spec['name']}.npy", oof); np.save(ARTIFACT_DIR / f"test_{spec['name']}.npy", TEST[spec["name"]])


######## main_tiny_ftune (3class) ########
fold 0 | train 169
  epoch 01/25 loss 1.0375
  epoch 02/25 loss 0.8880
  epoch 03/25 loss 0.7588
  epoch 04/25 loss 0.6858
  epoch 05/25 loss 0.6016
  epoch 06/25 loss 0.5694
  epoch 07/25 loss 0.5468
  epoch 08/25 loss 0.4893
  epoch 09/25 loss 0.4248
  epoch 10/25 loss 0.5000
  epoch 11/25 loss 0.4233
  epoch 12/25 loss 0.4567
  epoch 13/25 loss 0.3877
  epoch 14/25 loss 0.3824
  epoch 15/25 loss 0.4121
  epoch 16/25 loss 0.3969
  epoch 17/25 loss 0.3925
  epoch 18/25 loss 0.3462
  epoch 19/25 loss 0.3689
  epoch 20/25 loss 0.3475
  epoch 21/25 loss 0.3711
  epoch 22/25 loss 0.3368
  epoch 23/25 loss 0.3588
  epoch 24/25 loss 0.4067
  epoch 25/25 loss 0.3829
  fold 0 EMA macro F1 0.7557
fold 1 | train 169
  epoch 01/25 loss 1.1029
  epoch 02/25 loss 0.9181
  epoch 03/25 loss 0.7815
  epoch 04/25 loss 0.6483
  epoch 05/25 loss 0.5919
  epoch 06/25 loss 0.5313
  epoch 07/25 loss 0.4717
  epoch 08/25 loss 0.5131
  epoch 09/25 loss 0.4823
  epo

## 8. Fold tetap 5 lipat

In [11]:
EXT_NAMES = []
prob_cols = [f"prob_{n}" for n in CLASS_NAMES]
for d in EXTERNAL_ARTIFACT_DIRS:
    for oof_path in sorted(Path(d).glob("oof_*.csv")):
        name = "ext_" + oof_path.stem[len("oof_"):]
        test_path = Path(d) / f"test_{oof_path.stem[len('oof_'):]}.csv"
        if not test_path.is_file():
            continue
        o = pd.read_csv(oof_path); o["image_id"] = o["image_id"].astype(str); o = o.set_index("image_id")
        t = pd.read_csv(test_path); t["image_id"] = t["image_id"].astype(str); t = t.set_index("image_id")
        if not set(train_df["image_id"]) <= set(o.index) or not set(TEST_IDS) <= set(t.index):
            print("lewati", name, "karena image_id tidak lengkap"); continue
        if "fold" in o.columns and not np.array_equal(o.loc[train_df["image_id"], "fold"].to_numpy(), train_df["fold"].to_numpy()):
            print("PERINGATAN", name, "pembagian fold berbeda, stacking untuk anggota ini tidak sepenuhnya jujur")
        OOF[name] = o.loc[train_df["image_id"], prob_cols].to_numpy(dtype=np.float32)
        TEST[name] = t.loc[TEST_IDS, prob_cols].to_numpy(dtype=np.float32)
        EXT_NAMES.append(name)
        print(name, "OOF macro F1", round(macro(Y_TRAIN, OOF[name]), 4))
print("anggota eksternal", EXT_NAMES)


anggota eksternal []


## 9. Metrik, EMA, optimizer LLRD, scheduler, loss

In [12]:
@torch.inference_mode()
def embed_tiles(encoder, ids):
    encoder.eval(); out = {}
    for iid in tqdm(ids, desc="Embed tile"):
        x = torch.stack([to_tensor(t) for t in cache_tiles[iid]]).to(DEVICE)
        with ac():
            e = encoder(x).float()
        out[iid] = e.cpu().numpy()
    return out


tile_encoder = load_encoder(TILE_ENCODER["model"], TILE_ENCODER["init"]).to(DEVICE)
ALL_IDS = train_df["image_id"].tolist() + TEST_IDS
tile_emb = embed_tiles(tile_encoder, ALL_IDS)
del tile_encoder; torch.cuda.empty_cache()

E_train = np.stack([tile_emb[i] for i in train_df["image_id"]]).astype(np.float32)      # N x 12 x 768
E_test = np.stack([tile_emb[i] for i in TEST_IDS]).astype(np.float32)
M_train = np.stack([tile_fracs[i] for i in train_df["image_id"]]) > 0.05
M_test = np.stack([tile_fracs[i] for i in TEST_IDS]) > 0.05


class AttnMIL(nn.Module):
    def __init__(self, d=EMBED_DIM, h=128, n_classes=3, p=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.attn = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Dropout(p), nn.Linear(h, 1))
        self.head = nn.Sequential(nn.Dropout(p), nn.Linear(d, n_classes))

    def forward(self, x, mask):
        x = self.norm(x)
        a = self.attn(x).squeeze(-1).masked_fill(~mask, -1e4)
        a = torch.softmax(a, 1).unsqueeze(-1)
        return self.head((a * x).sum(1))


def train_mil(Xtr, Mtr, ytr, Xev, Mev, seed, epochs=300, lr=1e-3):
    seed_everything(seed)
    net = AttnMIL().to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-2)
    cnt = np.bincount(ytr, minlength=3).astype(float); w = torch.tensor((1 / cnt) / (1 / cnt).mean(), dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    Xtr_t, Mtr_t, ytr_t = torch.tensor(Xtr, device=DEVICE), torch.tensor(Mtr, device=DEVICE), torch.tensor(ytr, device=DEVICE)
    for ep in range(epochs):
        net.train(); idx = torch.randperm(len(Xtr_t), device=DEVICE)
        for b in idx.split(32):
            xb = Xtr_t[b] + 0.02 * torch.randn_like(Xtr_t[b])   # noise ringan pada embedding
            loss = crit(net(xb, Mtr_t[b]), ytr_t[b]); opt.zero_grad(); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return torch.softmax(net(torch.tensor(Xev, device=DEVICE), torch.tensor(Mev, device=DEVICE)), 1).cpu().numpy()


oof_mil = np.full((N_TRAIN, 3), np.nan, dtype=np.float32); test_mil = []
for f in FOLDS:
    tr, va = (train_df["fold"] != f).to_numpy(), (train_df["fold"] == f).to_numpy()
    both = np.concatenate([E_train[va], E_test]); both_m = np.concatenate([M_train[va], M_test])
    p = train_mil(E_train[tr], M_train[tr], Y_TRAIN[tr], both, both_m, SEED + f)
    oof_mil[va] = p[:va.sum()]; test_mil.append(p[va.sum():])
    print(f"fold {f} MIL macro F1 {macro(Y_TRAIN[va], oof_mil[va]):.4f}")
OOF["tile_mil"], TEST["tile_mil"] = oof_mil, np.mean(test_mil, axis=0)

Embed tile:   0%|          | 0/266 [00:00<?, ?it/s]

fold 0 MIL macro F1 0.5805
fold 1 MIL macro F1 0.6464
fold 2 MIL macro F1 0.4454
fold 3 MIL macro F1 0.4206
fold 4 MIL macro F1 0.4970


## 10. Fungsi training dan inferensi dengan TTA

In [13]:
main_names = [s["name"] for s in RUN_SPECS if s["task"] == "3class"] + EXT_NAMES
spec_names = [s["name"] for s in RUN_SPECS if s["task"] == "bvm"]
valid = ~np.isnan(OOF[main_names[0]]).any(1)
Y = Y_TRAIN[valid]

candidates = {}
for n in main_names:
    candidates[n] = (OOF[n][valid], TEST[n])
candidates["tile_mil"] = (OOF["tile_mil"][valid], TEST["tile_mil"])

avg_oof = np.mean([OOF[n][valid] for n in main_names], axis=0); avg_test = np.mean([TEST[n] for n in main_names], axis=0)
candidates["avg_main"] = (avg_oof, avg_test)

if spec_names:
    def hier(p3, pb):
        pn = p3[:, [0]]; pab = 1 - pn
        return np.concatenate([pn, pab * pb[:, [1]], pab * pb[:, [0]]], axis=1)   # pb[:,1] = P(Benign)
    sp_oof = np.mean([OOF[n][valid] for n in spec_names], axis=0); sp_test = np.mean([TEST[n] for n in spec_names], axis=0)
    candidates["hier"] = (hier(avg_oof, sp_oof), hier(avg_test, sp_test))
    candidates["avg_main_hier_mil"] = (np.mean([avg_oof, candidates["hier"][0], OOF["tile_mil"][valid]], axis=0),
                                       np.mean([avg_test, candidates["hier"][1], TEST["tile_mil"]], axis=0))


def meta_features(use_tab):
    cols_oof = [np.log(np.clip(OOF[n][valid], 1e-6, 1)) for n in main_names + spec_names + ["tile_mil"]]
    cols_test = [np.log(np.clip(TEST[n], 1e-6, 1)) for n in main_names + spec_names + ["tile_mil"]]
    Xo, Xt = np.concatenate(cols_oof, 1), np.concatenate(cols_test, 1)
    if use_tab:
        Xo = np.concatenate([Xo, tab_df.loc[train_df["image_id"][valid]].to_numpy()], 1)
        Xt = np.concatenate([Xt, tab_df.loc[TEST_IDS].to_numpy()], 1)
    return Xo, Xt


def stack(use_tab, C=0.3):
    Xo, Xt = meta_features(use_tab)
    folds = train_df["fold"].to_numpy()[valid]
    oof_s = np.zeros((len(Y), 3))
    for f in np.unique(folds):
        tr, va = folds != f, folds == f
        sc = StandardScaler().fit(Xo[tr])
        lr = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(sc.transform(Xo[tr]), Y[tr])
        oof_s[va] = lr.predict_proba(sc.transform(Xo[va]))
    sc = StandardScaler().fit(Xo)
    lr = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(sc.transform(Xo), Y)
    return oof_s, lr.predict_proba(sc.transform(Xt))


def tune_bias(po, folds, grid=np.linspace(-1.0, 1.0, 21), rounds=2):
    # bias aditif per kelas pada log probabilitas, dicari dengan koordinat, dinilai leave one fold out
    logp = np.log(np.clip(po, 1e-6, 1))

    def search(mask):
        b = np.zeros(3)
        for _ in range(rounds):
            for k in (1, 2):   # Normal jadi acuan, hanya bias Benign dan Malignant
                best_v, best_s = b[k], -1
                for v in grid:
                    bb = b.copy(); bb[k] = v
                    sc = f1_score(Y[mask], (logp[mask] + bb).argmax(1), average="macro", zero_division=0)
                    if sc > best_s + 1e-9:
                        best_s, best_v = sc, v
                b[k] = best_v
        return b

    cv_pred = np.zeros(len(Y), dtype=int); biases = []
    for f in np.unique(folds):
        b = search(folds != f); biases.append(b)
        cv_pred[folds == f] = (logp[folds == f] + b).argmax(1)
    return cv_pred, np.mean(biases, axis=0)


def with_bias(name):
    po, pt = candidates[name]
    folds = train_df["fold"].to_numpy()[valid]
    cv_pred, b = tune_bias(po, folds)
    oof_b = np.eye(3)[cv_pred]                                   # prediksi CV jujur, dipakai untuk skor
    pt_b = np.exp(np.log(np.clip(pt, 1e-6, 1)) + b); pt_b /= pt_b.sum(1, keepdims=True)
    print(f"bias {name}", b.round(2))
    return oof_b, pt_b


if len(FOLDS) > 1:
    for nm in ["avg_main", "hier"]:
        if nm in candidates:
            candidates[nm + "_bias"] = with_bias(nm)
    candidates["stack_probs"] = stack(False)
    candidates["stack_probs_tab"] = stack(True)

bias avg_main [ 0.   -0.38 -0.04]
bias hier [ 0.  -0.2  0. ]


## 11. Cross validation 5 lipat

Bobot yang dipakai untuk OOF dan test adalah EMA di akhir jadwal (tanpa pemilihan epoch terbaik). Log per epoch tetap ditampilkan untuk melihat dinamika, tetapi tidak dipakai untuk memilih bobot.

In [14]:
rng = np.random.default_rng(SEED)
rows = []
for name, (po, _) in candidates.items():
    pred = po.argmax(1)
    boots = [f1_score(Y[s], pred[s], average="macro", zero_division=0) for s in (rng.integers(0, len(Y), len(Y)) for _ in range(1000))]
    per = f1_score(Y, pred, labels=[0, 1, 2], average=None, zero_division=0)
    rows.append({"candidate": name, "macro_f1": macro(Y, po), "ci_lo": np.percentile(boots, 2.5), "ci_hi": np.percentile(boots, 97.5),
                 "f1_normal": per[0], "f1_benign": per[1], "f1_malignant": per[2]})
summary = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(summary.round(4).to_string())

best = summary.loc[0, "candidate"]
print(f"\nTerbaik menurut OOF: {best}")
print(confusion_matrix(Y, candidates[best][0].argmax(1)))
print(classification_report(Y, candidates[best][0].argmax(1), target_names=CLASS_NAMES, digits=4, zero_division=0))


def write_sub(name, pt):
    sub = pd.DataFrame({"image_id": TEST_IDS, "label": [ID_TO_LABEL[i] for i in pt.argmax(1)]})
    sub.to_csv(OUTPUT_DIR / f"submission_{name}.csv", index=False)
    print(f"submission_{name}.csv", sub["label"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict())


for name in dict.fromkeys([best, "avg_main", "hier", "hier_bias", "avg_main_hier_mil", "stack_probs_tab"]):
    if name in candidates:
        write_sub(name, candidates[name][1])
summary.to_csv(ARTIFACT_DIR / "summary.csv", index=False)

           candidate  macro_f1   ci_lo   ci_hi  f1_normal  f1_benign  f1_malignant
0          hier_bias    0.7368  0.6739  0.7983     0.8047     0.6355        0.7703
1        stack_probs    0.7281  0.6663  0.7836     0.7702     0.6491        0.7651
2  avg_main_hier_mil    0.7203  0.6581  0.7794     0.7871     0.6290        0.7448
3    main_tiny_ftune    0.7122  0.6496  0.7708     0.8000     0.6066        0.7299
4           avg_main    0.7122  0.6518  0.7741     0.8000     0.6066        0.7299
5      avg_main_bias    0.7111  0.6438  0.7706     0.7929     0.5818        0.7586
6               hier    0.7008  0.6407  0.7597     0.8000     0.5882        0.7143
7    stack_probs_tab    0.6648  0.5976  0.7281     0.7451     0.5246        0.7248
8           tile_mil    0.5217  0.4507  0.5877     0.5734     0.4511        0.5405

Terbaik menurut OOF: hier_bias
[[68  7  5]
 [12 34  6]
 [ 9 14 57]]
              precision    recall  f1-score   support

      Normal     0.7640    0.8500    0.8047   

## 12. Evaluasi OOF dan interval bootstrap

In [ ]:
evaluated = np.flatnonzero(~np.isnan(oof_full).any(axis=1))
oof_probs = oof_full[evaluated]
oof_targets = train_df.loc[evaluated, "target"].to_numpy()
oof_metrics = calculate_metrics(oof_targets, oof_probs)
fold_scores = np.array([m["macro_f1"] for m in fold_metrics])

rng = np.random.default_rng(SEED)
boot = []
for _ in range(2000):
    s = rng.integers(0, len(oof_targets), len(oof_targets))
    boot.append(f1_score(oof_targets[s], oof_probs[s].argmax(1), average="macro", zero_division=0))
ci = np.percentile(boot, [2.5, 97.5])

print("Global OOF macro F1", round(oof_metrics["macro_f1"], 4), "| 95% CI", ci.round(4))
print("Fold mean +/- std", round(fold_scores.mean(), 4), round(fold_scores.std(), 4))
print("F1 per kelas", {k: round(v, 4) for k, v in oof_metrics["f1_per_class"].items()})
print(classification_report(oof_targets, oof_probs.argmax(1), target_names=CLASS_NAMES, digits=4, zero_division=0))
print("Confusion matrix (baris = true)"); print(np.array(oof_metrics["confusion_matrix"]))

if history:
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(figsize=(8, 4))
    for f, g in h.groupby("fold"):
        ax.plot(g["epoch"], g["ema_macro_f1"], marker="o", label=f"fold {f}")
    ax.set_xlabel("epoch"); ax.set_ylabel("EMA val macro F1"); ax.legend(); plt.show()

oof_frame = train_df.loc[evaluated, ["image_id", "label"]].copy()
for i, n in enumerate(CLASS_NAMES):
    oof_frame[f"prob_{n}"] = oof_probs[:, i]
oof_frame["fold"] = fold_assignment[evaluated]
oof_frame.to_csv(ARTIFACT_DIR / f"oof_{RUN_NAME}.csv", index=False)

## 13. Kalibrasi suhu dan koreksi prior shift

Dua alat dipakai.

1. Temperature scaling dengan suhu T yang di fit pada OOF, supaya probabilitas tidak terlalu percaya diri sebelum diestimasi.
2. Estimasi prior test secara unsupervised dengan EM (Saerens, Latinne, Decaestecker 2002). Ini sah di kompetisi sungguhan.

Catatan penting dari simulasi. EM mengandalkan asumsi model terkalibrasi dan tidak bias antar domain. Jika model di test condong ke satu kelas (misalnya banyak memprediksi Malignant karena citra test terlihat berbeda), EM justru bisa memperkuat kecondongan itu. Karena itu hasil EM selalu dibandingkan dengan argmax mentah, dan jika ground truth tersedia, dengan prior oracle sebagai batas atas.

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.optimize import linear_sum_assignment

test_probs_raw = np.mean(np.stack(test_fold_probs), axis=0)
assert np.allclose(test_probs_raw.sum(1), 1.0, atol=1e-4)


def apply_temperature(probs, T):
    logits = np.log(np.clip(probs, 1e-9, 1.0)) / T
    logits -= logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)


def fit_temperature(probs, targets):
    def nll(T):
        p = apply_temperature(probs, T)
        return -np.mean(np.log(np.clip(p[np.arange(len(targets)), targets], 1e-9, 1.0)))
    return float(minimize_scalar(nll, bounds=(0.3, 5.0), method="bounded").x)


def adjust_by_prior(probs, source_prior, target_prior, strength=1.0):
    adj = probs * ((target_prior / source_prior) ** strength)[None, :]
    return adj / adj.sum(axis=1, keepdims=True)


def estimate_prior_em(probs, source_prior, max_iter=500, tol=1e-7, floor=0.03):
    prior = source_prior.copy()
    for _ in range(max_iter):
        new_prior = adjust_by_prior(probs, source_prior, prior).mean(axis=0)
        new_prior = np.maximum(new_prior, floor); new_prior /= new_prior.sum()
        if np.abs(new_prior - prior).max() < tol:
            return new_prior
        prior = new_prior
    return prior


def count_matched_prediction(probs, target_prior):
    # Prediksi yang jumlah per kelasnya persis mengikuti prior, dipilih lewat assignment berbiaya minimum.
    n = len(probs)
    counts = np.floor(target_prior * n).astype(int)
    while counts.sum() < n:
        counts[np.argmax(target_prior * n - counts)] += 1
    cols = np.concatenate([np.full(c, k) for k, c in enumerate(counts)])
    cost = -np.log(np.clip(probs[:, cols], 1e-9, 1.0))
    rows, assigned = linear_sum_assignment(cost)
    pred = np.zeros(n, dtype=int); pred[rows] = cols[assigned]
    return pred


TEMPERATURE = fit_temperature(oof_probs, oof_targets)
oof_probs_cal = apply_temperature(oof_probs, TEMPERATURE)
test_probs_cal = apply_temperature(test_probs_raw, TEMPERATURE)

em_prior_oof = estimate_prior_em(oof_probs_cal, SOURCE_PRIOR)
em_prior_test = estimate_prior_em(test_probs_cal, SOURCE_PRIOR)
test_probs_em = adjust_by_prior(test_probs_cal, SOURCE_PRIOR, em_prior_test)

print("Suhu T (fit OOF)       ", round(TEMPERATURE, 3))
print("Prior sumber (efektif) ", SOURCE_PRIOR.round(3))
print("Prior train empiris    ", TRAIN_PRIOR.round(3))
print("EM pada OOF (sanity)   ", em_prior_oof.round(3), " seharusnya dekat prior train empiris")
print("EM pada test           ", em_prior_test.round(3))
print("Distribusi argmax mentah", np.bincount(test_probs_raw.argmax(1), minlength=3))
print("Distribusi argmax EM    ", np.bincount(test_probs_em.argmax(1), minlength=3))

## 14. Submission dan artefak

Tiga submission dibuat. `raw` adalah argmax mentah, `em` memakai prior hasil EM, dan `oracle` (hanya jika ground truth ada) memakai prior dari ground truth dengan pencocokan jumlah per kelas. Submission oracle hanya untuk latihan karena memakai distribusi label test.

In [ ]:
def make_submission(pred_ids, path):
    sub = pd.DataFrame({"image_id": sample_df["image_id"].astype(str), "label": [ID_TO_LABEL[i] for i in pred_ids]})
    assert len(sub) == len(sample_df) and set(sub["label"]) <= set(CLASS_NAMES)
    sub.to_csv(path, index=False)
    print(path.name, sub["label"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict())
    return sub


make_submission(test_probs_raw.argmax(1), OUTPUT_DIR / f"submission_{RUN_NAME}_raw.csv")
make_submission(test_probs_em.argmax(1), OUTPUT_DIR / f"submission_{RUN_NAME}_em.csv")

TRUE_TEST_PRIOR = None
if GROUNDTRUTH_PATH is not None:
    gt = pd.read_csv(GROUNDTRUTH_PATH)
    gt["image_id"] = gt["image_id"].astype(str)
    gt = gt.set_index("image_id").loc[sample_df["image_id"].astype(str)]
    y_test = gt["label"].map(LABEL_TO_ID).to_numpy()
    TRUE_TEST_PRIOR = np.bincount(y_test, minlength=NUM_CLASSES) / len(y_test)
    make_submission(count_matched_prediction(test_probs_cal, TRUE_TEST_PRIOR), OUTPUT_DIR / f"submission_{RUN_NAME}_oracle.csv")

tp_frame = pd.DataFrame({"image_id": sample_df["image_id"].astype(str)})
for i, n in enumerate(CLASS_NAMES):
    tp_frame[f"prob_{n}"] = test_probs_raw[:, i]
tp_frame.to_csv(ARTIFACT_DIR / f"test_{RUN_NAME}.csv", index=False)

json.dump({"run": RUN_NAME, "oof_macro_f1": oof_metrics["macro_f1"], "oof_ci95": ci.tolist(),
           "fold_macro_f1": fold_scores.tolist(), "f1_per_class": oof_metrics["f1_per_class"],
           "temperature": TEMPERATURE, "em_prior_test": em_prior_test.tolist(),
           "config": {"model": VISION_MODEL_NAME, "init": INIT_MODE, "epochs": epochs_to_run, "base_lr": BASE_LR,
                      "llrd": LLRD, "mixup_prob": MIXUP_PROB, "ema": EMA_DECAY, "tta": TTA_HFLIP}},
          open(ARTIFACT_DIR / f"metrics_{RUN_NAME}.json", "w"), indent=2)
print("Artefak tersimpan di", ARTIFACT_DIR)

## 15. Evaluasi terhadap ground truth (hanya jika file di attach, hanya untuk belajar)

Skenario A argmax mentah, B koreksi prior EM, C prior oracle lewat reweighting, D prior oracle lewat pencocokan jumlah. C dan D adalah batas atas dan tidak boleh dipakai di kompetisi nyata.

In [ ]:
def report_test(name, pred_ids):
    m = calculate_metrics(y_test, np.eye(NUM_CLASSES)[pred_ids])
    print(f"\n===== {name} =====")
    print("macro F1", round(m["macro_f1"], 4), "| F1 per kelas", {k: round(v, 3) for k, v in m["f1_per_class"].items()})
    print("prediksi", np.bincount(pred_ids, minlength=3), "| confusion (baris = true)")
    print(np.array(m["confusion_matrix"]))
    return m["macro_f1"]


if TRUE_TEST_PRIOR is not None:
    print("Prior test ground truth", TRUE_TEST_PRIOR.round(3), "| estimasi EM", em_prior_test.round(3))
    scores = {
        "A raw": report_test("A. argmax mentah", test_probs_raw.argmax(1)),
        "B EM": report_test("B. koreksi prior EM", test_probs_em.argmax(1)),
        "C oracle reweight": report_test("C. prior oracle, reweighting", adjust_by_prior(test_probs_cal, SOURCE_PRIOR, TRUE_TEST_PRIOR).argmax(1)),
        "D oracle count match": report_test("D. prior oracle, pencocokan jumlah", count_matched_prediction(test_probs_cal, TRUE_TEST_PRIOR)),
    }
    print("\nRingkasan", {k: round(v, 4) for k, v in scores.items()})
    print("Jika B lebih buruk dari A, model masih bias di domain test dan EM tidak boleh dipercaya untuk run ini.")
else:
    print("Ground truth tidak tersedia, evaluasi test dilewati.")

## 16. Blending beberapa run (opsional)

Jalankan notebook ini beberapa kali dengan `VISION_MODEL_NAME` dan `INIT_MODE` berbeda (misalnya tiny ftune, tiny pretrain, small ftune), simpan folder `artifacts` masing masing sebagai Kaggle Dataset, lalu isi `BLEND_DIRS`. Bobot blend dicari hanya dari OOF dengan skema leave one fold out.

In [ ]:
BLEND_DIRS = [ARTIFACT_DIR]   # tambahkan Path ke folder artifacts run lain

runs = {}
for d in BLEND_DIRS:
    for oof_path in Path(d).glob("oof_*.csv"):
        name = oof_path.stem[len("oof_"):]
        test_path = Path(d) / f"test_{name}.csv"
        if test_path.is_file():
            runs[name] = (pd.read_csv(oof_path), pd.read_csv(test_path))
print("Run tersedia", list(runs))

if len(runs) >= 2:
    names = list(runs)
    base = runs[names[0]][0][["image_id", "label", "fold"]].copy()
    y = base["label"].map(LABEL_TO_ID).to_numpy()
    folds = base["fold"].to_numpy()
    prob_cols = [f"prob_{n}" for n in CLASS_NAMES]
    oof_stack = np.stack([runs[n][0].set_index("image_id").loc[base["image_id"], prob_cols].to_numpy() for n in names])
    test_stack = np.stack([runs[n][1].set_index("image_id").loc[sample_df["image_id"].astype(str), prob_cols].to_numpy() for n in names])

    rng = np.random.default_rng(SEED)
    cands = np.vstack([np.eye(len(names)), np.full((1, len(names)), 1.0 / len(names)), rng.dirichlet(np.ones(len(names)), size=300)])
    cross_pred = np.zeros(len(y), dtype=int); chosen = []
    for held in np.unique(folds):
        tune = folds != held
        best_w, best_s = None, -1
        for w in cands:
            s = f1_score(y[tune], np.tensordot(w, oof_stack[:, tune], axes=1).argmax(1), average="macro", zero_division=0)
            if s > best_s:
                best_s, best_w = s, w
        chosen.append(best_w)
        cross_pred[folds == held] = np.tensordot(best_w, oof_stack[:, folds == held], axes=1).argmax(1)
    final_w = np.mean(chosen, axis=0); final_w /= final_w.sum()
    print("Cross tuned OOF macro F1 blend", round(f1_score(y, cross_pred, average="macro", zero_division=0), 4))
    print("Bobot final", dict(zip(names, final_w.round(3))))

    blend_oof = np.tensordot(final_w, oof_stack, axes=1)
    blend_test = np.tensordot(final_w, test_stack, axes=1)
    T_blend = fit_temperature(blend_oof, y)
    blend_test_cal = apply_temperature(blend_test, T_blend)
    em_prior_blend = estimate_prior_em(blend_test_cal, SOURCE_PRIOR)
    print("OOF macro F1 bobot final", round(f1_score(y, blend_oof.argmax(1), average="macro", zero_division=0), 4), "| EM prior test", em_prior_blend.round(3))
    make_submission(blend_test.argmax(1), OUTPUT_DIR / "submission_blend_raw.csv")
    make_submission(adjust_by_prior(blend_test_cal, SOURCE_PRIOR, em_prior_blend).argmax(1), OUTPUT_DIR / "submission_blend_em.csv")
    if TRUE_TEST_PRIOR is not None:
        make_submission(count_matched_prediction(blend_test_cal, TRUE_TEST_PRIOR), OUTPUT_DIR / "submission_blend_oracle.csv")
        report_test("Blend raw", blend_test.argmax(1))
        report_test("Blend EM", adjust_by_prior(blend_test_cal, SOURCE_PRIOR, em_prior_blend).argmax(1))
        report_test("Blend oracle count match", count_matched_prediction(blend_test_cal, TRUE_TEST_PRIOR))
else:
    print("Butuh minimal 2 run untuk blending.")

## 17. Catatan

1. Pilih model dan hyperparameter hanya dari OOF. Ground truth di sel 15 adalah cermin untuk belajar, bukan target tuning.
2. EM hanya bisa dipercaya bila model terkalibrasi dan tidak bias di domain test. Cek sel 15. Jika skenario B lebih buruk dari A, perbaiki modelnya dulu (ensemble, augmentasi, data eksternal) sebelum mengandalkan EM.
3. Malignant di test sedikit, jadi satu salah tebak menggeser macro F1 beberapa poin. Terima variansnya.
4. Eksperimen lanjutan yang masuk akal. Ganti `INIT_MODE`, coba `convnext_small`, naikkan `MAX_EPOCHS` ke 40, atau pretraining tambahan dengan data mamografi publik (MammosighTR dari repo MAM CLIP, CBIS DDSM, VinDr Mammo) jika aturan lomba mengizinkan.